In [1]:
# =============================================================
# Fase 4 — Métricas de fragmentación del paisaje de manglar
# Pipeline multilenguaje CGSM (2018–2025)
# Lina María Quintero Fonseca | Maestría en Geomática, UNAL
# =============================================================

using DataFrames
using CSV
using GeoJSON
using Statistics

println("Fase 4 inicializada")

Fase 4 inicializada


In [2]:
# --- Cargar parches de manglar por período ---
using GeoJSON

periodos = ["degradacion", "recuperacion", "actual"]
base_dir = "/home/rstudio/work/proyecto-cgsm/data/processed/samgeo"

for periodo in periodos
    path = joinpath(base_dir, "manglar_$(periodo).geojson")
    geojson = GeoJSON.read(read(path, String))
    n = length(geojson.features)
    println("$(periodo): $(n) parches")
end

degradacion: 1548 parches
recuperacion: 1170 parches
actual: 488 parches


In [3]:
# --- Métricas de fragmentación del paisaje ---

function compute_metrics(periodo::String, base_dir::String)
    path = joinpath(base_dir, "manglar_$(periodo).geojson")
    geojson = GeoJSON.read(read(path, String))
    
    # Calcular área y perímetro de cada parche
    areas = Float64[]
    perimeters = Float64[]
    centroids_x = Float64[]
    centroids_y = Float64[]
    
    for feat in geojson.features
        geom = feat.geometry
        if geom isa GeoJSON.Polygon
            coords = geom.coordinates[1]  # anillo exterior
            n_pts = length(coords)
            
            # Área con fórmula del Shoelace (en grados, aprox)
            area = 0.0
            perim = 0.0
            cx, cy = 0.0, 0.0
            
            for i in 1:n_pts-1
                x1, y1 = coords[i]
                x2, y2 = coords[i+1]
                area += x1 * y2 - x2 * y1
                dx = (x2 - x1) * 111320 * cosd(y1)  # metros aprox
                dy = (y2 - y1) * 110540
                perim += sqrt(dx^2 + dy^2)
                cx += x1
                cy += y1
            end
            
            area = abs(area) / 2.0
            # Convertir área de grados² a hectáreas (aprox en lat 10.7°)
            area_ha = area * (111320 * cosd(10.7)) * 110540 / 10000
            cx /= (n_pts - 1)
            cy /= (n_pts - 1)
            
            # Filtrar: 1-5000 ha
            if area_ha >= 1.0 && area_ha < 5000.0
                push!(areas, area_ha)
                push!(perimeters, perim)
                push!(centroids_x, cx)
                push!(centroids_y, cy)
            end
        end
    end
    
    n_parches = length(areas)
    if n_parches == 0
        println("  $(periodo): sin parches válidos")
        return nothing
    end
    
    # Métricas
    area_total = sum(areas)
    area_media = mean(areas)
    area_std = std(areas)
    
    # Densidad de parches (parches por 1000 ha)
    densidad = n_parches / (area_total / 1000)
    
    # Índice de forma medio: MSI = perimetro / sqrt(π * area_m2)
    msi_values = Float64[]
    for i in 1:n_parches
        area_m2 = areas[i] * 10000
        msi = perimeters[i] / sqrt(π * area_m2)
        push!(msi_values, msi)
    end
    msi_mean = mean(msi_values)
    
    # Distancia al vecino más cercano (NND)
    nnd_values = Float64[]
    for i in 1:n_parches
        min_dist = Inf
        for j in 1:n_parches
            if i != j
                dx = (centroids_x[i] - centroids_x[j]) * 111320 * cosd(10.7)
                dy = (centroids_y[i] - centroids_y[j]) * 110540
                dist = sqrt(dx^2 + dy^2)
                if dist < min_dist
                    min_dist = dist
                end
            end
        end
        push!(nnd_values, min_dist)
    end
    nnd_mean = mean(nnd_values) / 1000  # en km
    
    println("  $(periodo):")
    println("    Parches: $(n_parches)")
    println("    Área total: $(round(area_total, digits=1)) ha")
    println("    Área media: $(round(area_media, digits=1)) ± $(round(area_std, digits=1)) ha")
    println("    Densidad: $(round(densidad, digits=2)) parches/1000 ha")
    println("    MSI medio: $(round(msi_mean, digits=2))")
    println("    NND medio: $(round(nnd_mean, digits=2)) km")
    
    return Dict(
        "periodo" => periodo,
        "n_parches" => n_parches,
        "area_total" => round(area_total, digits=1),
        "area_media" => round(area_media, digits=1),
        "area_std" => round(area_std, digits=1),
        "densidad" => round(densidad, digits=2),
        "msi_mean" => round(msi_mean, digits=2),
        "nnd_mean" => round(nnd_mean, digits=2)
    )
end

println("=== MÉTRICAS DE FRAGMENTACIÓN ===\n")
results = []
for periodo in ["degradacion", "recuperacion", "actual"]
    r = compute_metrics(periodo, base_dir)
    if r !== nothing
        push!(results, r)
    end
end

=== MÉTRICAS DE FRAGMENTACIÓN ===

  degradacion:
    Parches: 596
    Área total: 53283.4 ha
    Área media: 89.4 ± 81.6 ha
    Densidad: 11.19 parches/1000 ha
    MSI medio: 0.29
    NND medio: 0.46 km
  recuperacion:
    Parches: 456
    Área total: 47010.5 ha
    Área media: 103.1 ± 145.8 ha
    Densidad: 9.7 parches/1000 ha
    MSI medio: 0.31
    NND medio: 0.52 km
  actual:
    Parches: 183
    Área total: 28265.3 ha
    Área media: 154.5 ± 313.2 ha
    Densidad: 6.47 parches/1000 ha
    MSI medio: 0.74
    NND medio: 1.24 km


In [6]:
# --- Guardar tabla comparativa ---
df = DataFrame(
    Periodo = [r["periodo"] for r in results],
    Parches = [r["n_parches"] for r in results],
    Area_total_ha = [r["area_total"] for r in results],
    Area_media_ha = [r["area_media"] for r in results],
    Area_std_ha = [r["area_std"] for r in results],
    Densidad_parches_1000ha = [r["densidad"] for r in results],
    MSI_medio = [r["msi_mean"] for r in results],
    NND_medio_km = [r["nnd_mean"] for r in results]
)

CSV.write("../outputs/tables/metricas_fragmentacion.csv", df)
println("Tabla guardada en outputs/tables/metricas_fragmentacion.csv\n")
println(df)

Tabla guardada en outputs/tables/metricas_fragmentacion.csv

3×8 DataFrame
 Row │ Periodo       Parches  Area_total_ha  Area_media_ha  Area_std_ha  Densidad_parches_1000ha  MSI_medio  NND_medio_km 
     │ String        Int64    Float64        Float64        Float64      Float64                  Float64    Float64      
─────┼────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
   1 │ degradacion       596        53283.4           89.4         81.6                    11.19       0.29          0.46
   2 │ recuperacion      456        47010.5          103.1        145.8                     9.7        0.31          0.52
   3 │ actual            183        28265.3          154.5        313.2                     6.47       0.74          1.24


In [7]:
# --- Análisis detallado por período ---

println("=== DISTRIBUCIÓN DE ÁREAS POR PERÍODO ===\n")

function detailed_analysis(periodo::String, base_dir::String)
    path = joinpath(base_dir, "manglar_$(periodo).geojson")
    geojson = GeoJSON.read(read(path, String))
    
    areas = Float64[]
    perimeters = Float64[]
    centroids_x = Float64[]
    centroids_y = Float64[]
    
    for feat in geojson.features
        geom = feat.geometry
        if geom isa GeoJSON.Polygon
            coords = geom.coordinates[1]
            n_pts = length(coords)
            area = 0.0
            perim = 0.0
            cx = 0.0
            cy = 0.0
            
            for i in 1:n_pts-1
                x1, y1 = coords[i]
                x2, y2 = coords[i+1]
                area += x1 * y2 - x2 * y1
                dx = (x2 - x1) * 111320 * cosd(y1)
                dy = (y2 - y1) * 110540
                perim += sqrt(dx^2 + dy^2)
                cx += x1
                cy += y1
            end
            
            area = abs(area) / 2.0
            area_ha = area * (111320 * cosd(10.7)) * 110540 / 10000
            cx = cx / (n_pts - 1)
            cy = cy / (n_pts - 1)
            
            if area_ha >= 1.0 && area_ha < 5000.0
                push!(areas, area_ha)
                push!(perimeters, perim)
                push!(centroids_x, cx)
                push!(centroids_y, cy)
            end
        end
    end
    
    return areas, perimeters, centroids_x, centroids_y
end

# Cargar datos de los tres períodos
all_areas = Dict{String, Vector{Float64}}()
all_perimeters = Dict{String, Vector{Float64}}()
all_cx = Dict{String, Vector{Float64}}()
all_cy = Dict{String, Vector{Float64}}()

for periodo in ["degradacion", "recuperacion", "actual"]
    a, p, cx, cy = detailed_analysis(periodo, base_dir)
    all_areas[periodo] = a
    all_perimeters[periodo] = p
    all_cx[periodo] = cx
    all_cy[periodo] = cy
end

# --- Distribución por clases de tamaño ---
println("Distribución por clases de tamaño:\n")
println(lpad("Clase (ha)", 15), " | ",
        lpad("Degradación", 12), " | ",
        lpad("Recuperación", 12), " | ",
        lpad("Actual", 12))
println("-"^55)

clases = [(1, 10, "1-10"), (10, 50, "10-50"), (50, 100, "50-100"),
          (100, 500, "100-500"), (500, 1000, "500-1000"), (1000, 5000, "1000-5000")]

for (lo, hi, label) in clases
    c1 = count(a -> a >= lo && a < hi, all_areas["degradacion"])
    c2 = count(a -> a >= lo && a < hi, all_areas["recuperacion"])
    c3 = count(a -> a >= lo && a < hi, all_areas["actual"])
    println(lpad(label, 15), " | ",
            lpad(string(c1), 12), " | ",
            lpad(string(c2), 12), " | ",
            lpad(string(c3), 12))
end

# --- Estadísticas descriptivas ---
println("\n=== ESTADÍSTICAS DESCRIPTIVAS DE ÁREA (ha) ===\n")
println(lpad("Estadístico", 15), " | ",
        lpad("Degradación", 12), " | ",
        lpad("Recuperación", 12), " | ",
        lpad("Actual", 12))
println("-"^55)

stats_names = ["N parches", "Mínimo", "Q1 (25%)", "Mediana", "Q3 (75%)", "Máximo", "Media", "Desv. std.", "Suma total"]

for periodo_label in ["degradacion", "recuperacion", "actual"]
    # solo para construir la tabla después
end

# Calcular stats por período
function calc_stats(areas)
    return Dict(
        "N parches" => length(areas),
        "Mínimo" => minimum(areas),
        "Q1 (25%)" => quantile(areas, 0.25),
        "Mediana" => median(areas),
        "Q3 (75%)" => quantile(areas, 0.75),
        "Máximo" => maximum(areas),
        "Media" => mean(areas),
        "Desv. std." => std(areas),
        "Suma total" => sum(areas)
    )
end

s_deg = calc_stats(all_areas["degradacion"])
s_rec = calc_stats(all_areas["recuperacion"])
s_act = calc_stats(all_areas["actual"])

for sn in stats_names
    v1 = round(s_deg[sn], digits=1)
    v2 = round(s_rec[sn], digits=1)
    v3 = round(s_act[sn], digits=1)
    println(lpad(sn, 15), " | ",
            lpad(string(v1), 12), " | ",
            lpad(string(v2), 12), " | ",
            lpad(string(v3), 12))
end

# --- Conectividad ---
println("\n=== CONECTIVIDAD (Distancia al Vecino más Cercano) ===\n")

function calc_nnd(cx, cy)
    n = length(cx)
    nnd = Float64[]
    for i in 1:n
        min_dist = Inf
        for j in 1:n
            if i != j
                ddx = (cx[i] - cx[j]) * 111320 * cosd(10.7)
                ddy = (cy[i] - cy[j]) * 110540
                dist = sqrt(ddx^2 + ddy^2)
                if dist < min_dist
                    min_dist = dist
                end
            end
        end
        push!(nnd, min_dist / 1000)
    end
    return nnd
end

for periodo in ["degradacion", "recuperacion", "actual"]
    nnd = calc_nnd(all_cx[periodo], all_cy[periodo])
    n = length(nnd)
    aislados = count(d -> d > 5.0, nnd)
    
    println("$(periodo):")
    println("  NND mínimo:  $(round(minimum(nnd), digits=2)) km")
    println("  NND medio:   $(round(mean(nnd), digits=2)) km")
    println("  NND mediana: $(round(median(nnd), digits=2)) km")
    println("  NND máximo:  $(round(maximum(nnd), digits=2)) km")
    println("  Parches aislados (NND > 5 km): $(aislados) ($(round(100*aislados/n, digits=1))%)")
    println()
end

# --- Índice de forma ---
println("=== ÍNDICE DE FORMA (MSI) ===")
println("MSI = 1.0 → forma circular | MSI > 1.0 → forma irregular\n")

function calc_msi(areas, perimeters)
    msi = Float64[]
    for i in 1:length(areas)
        area_m2 = areas[i] * 10000
        m = perimeters[i] / sqrt(π * area_m2)
        push!(msi, m)
    end
    return msi
end

for periodo in ["degradacion", "recuperacion", "actual"]
    msi = calc_msi(all_areas[periodo], all_perimeters[periodo])
    
    compact = count(m -> m < 1.5, msi)
    moderate = count(m -> m >= 1.5 && m < 2.5, msi)
    complex_n = count(m -> m >= 2.5, msi)
    
    println("$(periodo):")
    println("  MSI mínimo: $(round(minimum(msi), digits=2))")
    println("  MSI medio:  $(round(mean(msi), digits=2))")
    println("  MSI máximo: $(round(maximum(msi), digits=2))")
    println("  Compactos (MSI < 1.5):  $(compact)")
    println("  Moderados (1.5-2.5):    $(moderate)")
    println("  Complejos (MSI > 2.5):  $(complex_n)")
    println()
end

=== DISTRIBUCIÓN DE ÁREAS POR PERÍODO ===

Distribución por clases de tamaño:

     Clase (ha) |  Degradación | Recuperación |       Actual
-------------------------------------------------------
           1-10 |            0 |            0 |            0
          10-50 |            0 |            0 |            0
         50-100 |          554 |          420 |          157
        100-500 |           36 |           25 |           14
       500-1000 |            5 |            8 |            5
      1000-5000 |            1 |            3 |            7

=== ESTADÍSTICAS DESCRIPTIVAS DE ÁREA (ha) ===

    Estadístico |  Degradación | Recuperación |       Actual
-------------------------------------------------------
      N parches |        596.0 |        456.0 |        183.0
         Mínimo |         73.8 |         73.8 |         73.8
       Q1 (25%) |         73.8 |         73.8 |         73.8
        Mediana |         73.8 |         73.8 |         73.8
       Q3 (75%) |         73

In [8]:
# --- Análisis detallado por período ---

println("=== DISTRIBUCIÓN DE ÁREAS POR PERÍODO ===\n")

function detailed_analysis(periodo::String, base_dir::String)
    path = joinpath(base_dir, "manglar_$(periodo).geojson")
    geojson = GeoJSON.read(read(path, String))
    
    areas = Float64[]
    perimeters = Float64[]
    centroids_x = Float64[]
    centroids_y = Float64[]
    
    for feat in geojson.features
        geom = feat.geometry
        if geom isa GeoJSON.Polygon
            coords = geom.coordinates[1]
            n_pts = length(coords)
            area = 0.0
            perim = 0.0
            cx = 0.0
            cy = 0.0
            
            for i in 1:n_pts-1
                x1, y1 = coords[i]
                x2, y2 = coords[i+1]
                area += x1 * y2 - x2 * y1
                dx = (x2 - x1) * 111320 * cosd(y1)
                dy = (y2 - y1) * 110540
                perim += sqrt(dx^2 + dy^2)
                cx += x1
                cy += y1
            end
            
            area = abs(area) / 2.0
            area_ha = area * (111320 * cosd(10.7)) * 110540 / 10000
            cx = cx / (n_pts - 1)
            cy = cy / (n_pts - 1)
            
            if area_ha >= 1.0 && area_ha < 5000.0
                push!(areas, area_ha)
                push!(perimeters, perim)
                push!(centroids_x, cx)
                push!(centroids_y, cy)
            end
        end
    end
    
    return areas, perimeters, centroids_x, centroids_y
end

# Cargar datos de los tres períodos
all_areas = Dict{String, Vector{Float64}}()
all_perimeters = Dict{String, Vector{Float64}}()
all_cx = Dict{String, Vector{Float64}}()
all_cy = Dict{String, Vector{Float64}}()

for periodo in ["degradacion", "recuperacion", "actual"]
    a, p, cx, cy = detailed_analysis(periodo, base_dir)
    all_areas[periodo] = a
    all_perimeters[periodo] = p
    all_cx[periodo] = cx
    all_cy[periodo] = cy
end

# --- Distribución por clases de tamaño ---
println("Distribución por clases de tamaño:\n")
println(lpad("Clase (ha)", 15), " | ",
        lpad("Degradación", 12), " | ",
        lpad("Recuperación", 12), " | ",
        lpad("Actual", 12))
println("-"^55)

clases = [(1, 10, "1-10"), (10, 50, "10-50"), (50, 100, "50-100"),
          (100, 500, "100-500"), (500, 1000, "500-1000"), (1000, 5000, "1000-5000")]

for (lo, hi, label) in clases
    c1 = count(a -> a >= lo && a < hi, all_areas["degradacion"])
    c2 = count(a -> a >= lo && a < hi, all_areas["recuperacion"])
    c3 = count(a -> a >= lo && a < hi, all_areas["actual"])
    println(lpad(label, 15), " | ",
            lpad(string(c1), 12), " | ",
            lpad(string(c2), 12), " | ",
            lpad(string(c3), 12))
end

# --- Estadísticas descriptivas ---
println("\n=== ESTADÍSTICAS DESCRIPTIVAS DE ÁREA (ha) ===\n")
println(lpad("Estadístico", 15), " | ",
        lpad("Degradación", 12), " | ",
        lpad("Recuperación", 12), " | ",
        lpad("Actual", 12))
println("-"^55)

stats_names = ["N parches", "Mínimo", "Q1 (25%)", "Mediana", "Q3 (75%)", "Máximo", "Media", "Desv. std.", "Suma total"]

for periodo_label in ["degradacion", "recuperacion", "actual"]
    # solo para construir la tabla después
end

# Calcular stats por período
function calc_stats(areas)
    return Dict(
        "N parches" => length(areas),
        "Mínimo" => minimum(areas),
        "Q1 (25%)" => quantile(areas, 0.25),
        "Mediana" => median(areas),
        "Q3 (75%)" => quantile(areas, 0.75),
        "Máximo" => maximum(areas),
        "Media" => mean(areas),
        "Desv. std." => std(areas),
        "Suma total" => sum(areas)
    )
end

s_deg = calc_stats(all_areas["degradacion"])
s_rec = calc_stats(all_areas["recuperacion"])
s_act = calc_stats(all_areas["actual"])

for sn in stats_names
    v1 = round(s_deg[sn], digits=1)
    v2 = round(s_rec[sn], digits=1)
    v3 = round(s_act[sn], digits=1)
    println(lpad(sn, 15), " | ",
            lpad(string(v1), 12), " | ",
            lpad(string(v2), 12), " | ",
            lpad(string(v3), 12))
end

# --- Conectividad ---
println("\n=== CONECTIVIDAD (Distancia al Vecino más Cercano) ===\n")

function calc_nnd(cx, cy)
    n = length(cx)
    nnd = Float64[]
    for i in 1:n
        min_dist = Inf
        for j in 1:n
            if i != j
                ddx = (cx[i] - cx[j]) * 111320 * cosd(10.7)
                ddy = (cy[i] - cy[j]) * 110540
                dist = sqrt(ddx^2 + ddy^2)
                if dist < min_dist
                    min_dist = dist
                end
            end
        end
        push!(nnd, min_dist / 1000)
    end
    return nnd
end

for periodo in ["degradacion", "recuperacion", "actual"]
    nnd = calc_nnd(all_cx[periodo], all_cy[periodo])
    n = length(nnd)
    aislados = count(d -> d > 5.0, nnd)
    
    println("$(periodo):")
    println("  NND mínimo:  $(round(minimum(nnd), digits=2)) km")
    println("  NND medio:   $(round(mean(nnd), digits=2)) km")
    println("  NND mediana: $(round(median(nnd), digits=2)) km")
    println("  NND máximo:  $(round(maximum(nnd), digits=2)) km")
    println("  Parches aislados (NND > 5 km): $(aislados) ($(round(100*aislados/n, digits=1))%)")
    println()
end

# --- Índice de forma ---
println("=== ÍNDICE DE FORMA (MSI) ===")
println("MSI = 1.0 → forma circular | MSI > 1.0 → forma irregular\n")

function calc_msi(areas, perimeters)
    msi = Float64[]
    for i in 1:length(areas)
        area_m2 = areas[i] * 10000
        m = perimeters[i] / sqrt(π * area_m2)
        push!(msi, m)
    end
    return msi
end

for periodo in ["degradacion", "recuperacion", "actual"]
    msi = calc_msi(all_areas[periodo], all_perimeters[periodo])
    
    compact = count(m -> m < 1.5, msi)
    moderate = count(m -> m >= 1.5 && m < 2.5, msi)
    complex_n = count(m -> m >= 2.5, msi)
    
    println("$(periodo):")
    println("  MSI mínimo: $(round(minimum(msi), digits=2))")
    println("  MSI medio:  $(round(mean(msi), digits=2))")
    println("  MSI máximo: $(round(maximum(msi), digits=2))")
    println("  Compactos (MSI < 1.5):  $(compact)")
    println("  Moderados (1.5-2.5):    $(moderate)")
    println("  Complejos (MSI > 2.5):  $(complex_n)")
    println()
end

=== DISTRIBUCIÓN DE ÁREAS POR PERÍODO ===

Distribución por clases de tamaño:

     Clase (ha) |  Degradación | Recuperación |       Actual
-------------------------------------------------------
           1-10 |            0 |            0 |            0
          10-50 |            0 |            0 |            0
         50-100 |          554 |          420 |          157
        100-500 |           36 |           25 |           14
       500-1000 |            5 |            8 |            5
      1000-5000 |            1 |            3 |            7

=== ESTADÍSTICAS DESCRIPTIVAS DE ÁREA (ha) ===

    Estadístico |  Degradación | Recuperación |       Actual
-------------------------------------------------------
      N parches |        596.0 |        456.0 |        183.0
         Mínimo |         73.8 |         73.8 |         73.8
       Q1 (25%) |         73.8 |         73.8 |         73.8
        Mediana |         73.8 |         73.8 |         73.8
       Q3 (75%) |         73